In [1]:
import pandas as pd
from transformers import pipeline

df = pd.read_csv('../data/jnt_reviews_cleaned.csv')

#Load model 
pretrained_name = "mdhugol/indonesia-bert-sentiment-classification"
nlp = pipeline("sentiment-analysis", model=pretrained_name, tokenizer=pretrained_name)

def get_sentiment_indobert(text):
    if not isinstance(text, str) or len(text) < 3:
        return 'neutral'
    
    result = nlp(text[:512])[0]
    
    label_map = {'LABEL_0': 'neutral', 'LABEL_1': 'positive', 'LABEL_2': 'negative'}
    return label_map.get(result['label'], 'neutral')

C:\Users\Sidqi\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Device set to use cuda:0


In [2]:
df['sentiment_bert'] = df['content_clean'].apply(get_sentiment_indobert)

# Grey Churn
def apply_churn_logic(row):
    if row['score'] <= 2:
        return 1
    elif row['score'] == 3 and row['sentiment_bert'] == 'negative':
        return 1
    else:
        return 0

df['is_churn'] = df.apply(apply_churn_logic, axis=1)

df.to_csv('../data/jnt_final_labeled.csv', index=False)

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
